# Notebook 02 - Random Baseline


## 2.1 Grundlagen

### Ziel
Dieses Notebook wertet die Random Baseline für die drei Unity-Environments aus.

### Methodik
Der Random Agent verwendet denselben Aktionsraum, dieselben Bewegungsparameter, dieselbe Spawn-/Goal-Randomisierung und dieselben Episode-Endbedingungen wie die später trainierten Agenten. Der Unterschied besteht darin, dass die Aktionen zufällig gewählt werden und kein Lernvorgang stattfindet.

### Experimentdesign
- Environments: Env1_Simple, Env2_Complex, Env3_Warehouse
- Seeds: 1, 27, 42, 72, 100
- Episoden pro Run: 200
- Max Steps pro Episode: 5000
- Action Space: Discrete(9)

---

Laden der Evaluationsdaten der Random Baseline und kruze check, dass die Daten gefunden wurden.

In [39]:
from pathlib import Path
import pandas as pd

log_dir = Path("../MazeAgent/training/evaluation_logs") # Pfad zum Verzeichnis mit den Log-Dateien der Random Baseline

files = sorted(log_dir.glob("Random_Env*.csv")) # Alle CSV-Dateien mit den Evaluations-Logs der Random Baseline auflisten und sortieren

len(files), files[:15] # Anzahl der gefundenen Dateien und die 15 Dateien anzeigen, die existeren sollten, um einen Überblick zu bekommen, ob alle erwarteten Dateien vorhanden sind

(15,
 [WindowsPath('../MazeAgent/training/evaluation_logs/Random_Env1_Seed1.csv'),
  WindowsPath('../MazeAgent/training/evaluation_logs/Random_Env1_Seed100.csv'),
  WindowsPath('../MazeAgent/training/evaluation_logs/Random_Env1_Seed27.csv'),
  WindowsPath('../MazeAgent/training/evaluation_logs/Random_Env1_Seed42.csv'),
  WindowsPath('../MazeAgent/training/evaluation_logs/Random_Env1_Seed72.csv'),
  WindowsPath('../MazeAgent/training/evaluation_logs/Random_Env2_Seed1.csv'),
  WindowsPath('../MazeAgent/training/evaluation_logs/Random_Env2_Seed100.csv'),
  WindowsPath('../MazeAgent/training/evaluation_logs/Random_Env2_Seed27.csv'),
  WindowsPath('../MazeAgent/training/evaluation_logs/Random_Env2_Seed42.csv'),
  WindowsPath('../MazeAgent/training/evaluation_logs/Random_Env2_Seed72.csv'),
  WindowsPath('../MazeAgent/training/evaluation_logs/Random_Env3_Seed1.csv'),
  WindowsPath('../MazeAgent/training/evaluation_logs/Random_Env3_Seed100.csv'),
  WindowsPath('../MazeAgent/training/evaluation

---

Betrachtung der ersten Zeilen für Env_1_Seed1, um die Struktur der Daten zu veranschaulichen und die Daten auch richtig wiedergegeben werden.

In [40]:
df = pd.concat([pd.read_csv(file) for file in files], ignore_index=True) # Alle CSV-Dateien in einem DataFrame zusammenführen, um eine umfassende Analyse der Random Baseline durchzuführen

df.head() # Anzeigen der ersten 5 Zeilen des zusammengeführten DataFrames, um einen Überblick über die Struktur und die enthaltenen Daten zu erhalten

,run_id,environment,algorithm,seed,episode,result,steps,total_reward,spawn_index,goal_index,max_steps,success,wall_hit,timeout,total_logged_steps
0,Random_Env1_Seed1,Env1_Simple,Random,1,1,Wall,184,-0.684,2,2,5000,0,1,0,184
1,Random_Env1_Seed1,Env1_Simple,Random,1,2,Wall,33,-0.533,3,3,5000,0,1,0,217
2,Random_Env1_Seed1,Env1_Simple,Random,1,3,Wall,280,-0.780,1,1,5000,0,1,0,497
3,Random_Env1_Seed1,Env1_Simple,Random,1,4,Wall,377,-0.877,2,3,5000,0,1,0,874
4,Random_Env1_Seed1,Env1_Simple,Random,1,5,Wall,209,-0.709,3,1,5000,0,1,0,1083


---

Laden des Shapes zur Überprüfung der Anzahl der gesammelten Datenpunkte 15 x 200 = 3000 Datenzeilen insgesamt.

In [41]:
df.shape # Überprüfen der Gesamtzahl der Zeilen und Spalten im DataFrame, um die Größe der gesammelten Daten zu verstehen

(3000, 15)

---

Zusammenfassung der Kernergebnisse der Random Baseline für die drei Environments, um eine Referenz für die spätere Bewertung der trainierten Agenten zu schaffen und hier erkennt man auchm dass 200 Episoden pro Run stattgefunden haben.

In [42]:
run_check = (
    df.groupby(["environment", "seed"]) # Gruppieren der Daten nach "environment" und "seed"
      .agg(
          episodes=("episode", "count"), # Zählen der Anzahl der Episoden pro Gruppe
          max_steps=("max_steps", "first"), # Übernehmen des Werts von "max_steps" aus der ersten Zeile jeder Gruppe, da dieser Wert für alle Zeilen in der Gruppe gleich sein sollte
          success_sum=("success", "sum"), # Summe der erfolgreichen Episoden pro Gruppe
          wall_sum=("wall_hit", "sum"), # Summe der Wandkollisionen pro Gruppe
          timeout_sum=("timeout", "sum"), # Summe der Zeitüberschreitungen pro Gruppe
      )
      .reset_index() # Zurücksetzen des Index, um die gruppierten Spalten wieder als reguläre Spalten im DataFrame zu haben
)

run_check

,environment,seed,episodes,max_steps,success_sum,wall_sum,timeout_sum
0,Env1_Simple,1,200,5000,0,200,0
1,Env1_Simple,27,200,5000,0,200,0
2,Env1_Simple,42,200,5000,0,200,0
3,Env1_Simple,72,200,5000,0,200,0
4,Env1_Simple,100,200,5000,0,200,0
5,Env2_Complex,1,200,5000,0,200,0
6,Env2_Complex,27,200,5000,0,200,0
7,Env2_Complex,42,200,5000,0,200,0
8,Env2_Complex,72,200,5000,0,200,0
9,Env2_Complex,100,200,5000,0,200,0


Interessanterweise hat es der Random Agent ganze 2 mal geschafft das Ziel zu erreichen, obwohl er nur "willkürliche" Aktionen ausgeführt hat. Das zeigt, dass es in dem Environment "Warehouse" auch ohne gezieltes Handeln möglich ist, das Ziel zu erreichen. (auch wenn die Wahrscheinlichkeiten dafür sehr gering sind.)

In [43]:
successful_runs = df[df["success"] == 1] # Filtern der Episoden, bei denen "success" gleich 1 ist, um die erfolgreichen Läufe zu analysieren
successful_runs

,run_id,environment,algorithm,seed,episode,result,steps,total_reward,spawn_index,goal_index,max_steps,success,wall_hit,timeout,total_logged_steps
2400,Random_Env3_Seed27,Env3_Warehouse,Random,27,1,Goal,159,4.841,1,0,5000,1,0,0,159
2793,Random_Env3_Seed42,Env3_Warehouse,Random,42,194,Goal,213,4.787,1,0,5000,1,0,0,22790


Interessant ist, dass wenn man die erfolgreichen Episoden betrachtet: Bei beiden "spawn_index = 1" und "goal_index = 0" zu erkennen ist. Schaut man in die Videodokumentation rein, sieht man sehr deutlich, dass die beiden Spawnpunkte sehr nah beieinander liegen, was die Wahrscheinlichkeit erhöht, dass der Agent zufällig das Ziel erreicht, es bleibt zu beobachten wie sich diese Tatsache auf die drei Algorithmen auswirkt, die trainiert werden.

Das Design wird nicht nochmal geändert, da es wichtig ist, die Random Baseline unter den gleichen Bedingungen wie die trainierten Agenten zu haben, um eine faire Vergleichsbasis zu gewährleisten. Außerdem ist wird es interessant zu beobachten sein, wie die trainierten Agenten mit dieser Tatsache umgehen, da sie ja auch unter diesen Bedingungen trainiert werden 

_Kleiner Hinweis an der Stelle: Bei der Erstellung der Spawn- und Goal-Punkte hatte ich nicht berücksichtigt, dass einige Punkte sehr nah beieinander liegen könnten, da ich gerade die Goal-Punkte im Environement "nach Gefühl" gesetzt habe. Gerade für zukünftige Experimente könnte es sinnvoll sein, die Punkte systematischer zu platzieren, um solche Zufallseffekte zu minimieren._

---

Betrachtung der Ergebnisse nach den 15 Seeds sortiert hinsichtlich des durchschnittlichen Rewards sowie der durchschnittlichen Schrittanzahl pro Episode. 

In [44]:
summary_seed = (
    df.groupby(["environment", "seed"]) # Gruppieren der Daten nach "environment" und "seed", um die Ergebnisse pro Umgebung und Seed zu analysieren
      .agg(
          episodes=("episode", "count"), # Zählen der Anzahl der Episoden pro Gruppe
          success_rate=("success", "mean"), # Berechnen der Erfolgsrate pro Gruppe, indem der Mittelwert der "success"-Spalte genommen wird (da "success" binär ist, entspricht dies dem Anteil der erfolgreichen Episoden)
          wall_rate=("wall_hit", "mean"), # Berechnen der Rate der Wandkollisionen pro Gruppe, indem der Mittelwert der "wall_hit"-Spalte genommen wird (da "wall_hit" binär ist, entspricht dies dem Anteil der Episoden mit Wandkollisionen)
          timeout_rate=("timeout", "mean"), # Berechnen der Rate der Zeitüberschreitungen pro Gruppe, indem der Mittelwert der "timeout"-Spalte genommen wird (da "timeout" binär ist, entspricht dies dem Anteil der Episoden mit Zeitüberschreitungen)
          mean_reward=("total_reward", "mean"), # Berechnen des durchschnittlichen Belohnungswerts pro Gruppe
          mean_steps=("steps", "mean"), # Berechnen der durchschnittlichen Anzahl der Schritte pro Gruppe
      )
      .reset_index() # Zurücksetzen des Index, um die gruppierten Spalten wieder als reguläre Spalten im DataFrame zu haben
)

summary_seed

,environment,seed,episodes,success_rate,wall_rate,timeout_rate,mean_reward,mean_steps
0,Env1_Simple,1,200,0.000,1.000,0.0,-1.009640,509.640
1,Env1_Simple,27,200,0.000,1.000,0.0,-1.036154,536.155
2,Env1_Simple,42,200,0.000,1.000,0.0,-1.049933,549.935
3,Env1_Simple,72,200,0.000,1.000,0.0,-0.942024,442.025
4,Env1_Simple,100,200,0.000,1.000,0.0,-0.951545,451.545
5,Env2_Complex,1,200,0.000,1.000,0.0,-0.626440,126.440
6,Env2_Complex,27,200,0.000,1.000,0.0,-0.618685,118.685
7,Env2_Complex,42,200,0.000,1.000,0.0,-0.590565,90.565
8,Env2_Complex,72,200,0.000,1.000,0.0,-0.616310,116.310
9,Env2_Complex,100,200,0.000,1.000,0.0,-0.612250,112.250


---

Betrachtung der Ergebnisse nach Environment, um zu sehen, ob es Unterschiede in der Erfolgsrate gibt

In [ ]:
summary_env = (
    summary_seed.groupby("environment") # Gruppieren der Daten nach "environment", um die Ergebnisse pro Umgebung zu analysieren
        .agg(
            success_rate_mean=("success_rate", "mean"), # Berechnen des durchschnittlichen Erfolgsraten pro Umgebung
            success_rate_std=("success_rate", "std"), # Berechnen der Standardabweichung der Erfolgsraten pro Umgebung
            wall_rate_mean=("wall_rate", "mean"), # Berechnen des durchschnittlichen Raten der Wandkollisionen pro Umgebung
            wall_rate_std=("wall_rate", "std"), # Berechnen der Standardabweichung der Raten der Wandkollisionen pro Umgebung
            timeout_rate_mean=("timeout_rate", "mean"), # Berechnen des durchschnittlichen Raten der Zeitüberschreitungen pro Umgebung
            timeout_rate_std=("timeout_rate", "std"), # Berechnen der Standardabweichung der Raten der Zeitüberschreitungen pro Umgebung
            mean_reward_mean=("mean_reward", "mean"), # Berechnen des durchschnittlichen Belohnungswerts pro Umgebung
            mean_reward_std=("mean_reward", "std"), # Berechnen der Standardabweichung der Belohnungswerte pro Umgebung
            mean_steps_mean=("mean_steps", "mean"), # Berechnen der durchschnittlichen Anzahl der Schritte pro Umgebung
            mean_steps_std=("mean_steps", "std"), # Berechnen der Standardabweichung der Anzahl der Schritte pro Umgebung
        )
        .reset_index() # Zurücksetzen des Index, um die gruppierten Spalten wieder als reguläre Spalten im DataFrame zu haben
)

summary_env

,environment,success_rate_mean,success_rate_std,wall_rate_mean,wall_rate_std,timeout_rate_mean,timeout_rate_std,mean_reward_mean,mean_reward_std,mean_steps_mean,mean_steps_std
0,Env1_Simple,0.000,0.000000,1.000,0.000000,0.0,0.0,-0.997859,0.048938,497.860,48.937990
1,Env2_Complex,0.000,0.000000,1.000,0.000000,0.0,0.0,-0.612850,0.013488,112.850,13.487848
2,Env3_Warehouse,0.002,0.002739,0.998,0.002739,0.0,0.0,-0.619269,0.020193,130.269,13.895854


---

Laden der Output Tabellen in den Docs-Ordner, um sie später in die Dokumentation einzubinden und im Vergleich zu den anderen Algorithmen auswerten zu können

In [45]:
output_dir = Path("../docs/evaluation/tables") # Pfad zum Verzeichnis, in dem die zusammengefassten Ergebnisse der Random Baseline gespeichert werden sollen
output_dir.mkdir(parents=True, exist_ok=True) # Erstellen des Verzeichnisses, falls es noch nicht existiert, einschließlich aller übergeordneten Verzeichnisse (parents=True) und ohne Fehler zu werfen, wenn das Verzeichnis bereits existiert (exist_ok=True)

summary_seed.to_csv(output_dir / "random_baseline_by_seed.csv", index=False) # Speichern der zusammengefassten Ergebnisse pro Seed als CSV-Datei im angegebenen Verzeichnis
summary_env.to_csv(output_dir / "random_baseline_by_environment.csv", index=False) # Speichern der zusammengefassten Ergebnisse pro Umgebung als CSV-Datei im angegebenen Verzeichnis

---

### Prüfung der Episoden mit Steps = 0

Bei der Durchsicht der Random-Baseline-Daten fällt auf, dass einzelne Episoden mit "steps = 0" geloggt wurden. Das bedeutet, dass die Episode beendet wurde, bevor der Agent einen regulären Action-Step ausführen konnte. Die genaue Ursache konnte im Rahmen der aktuellen Prüfung nicht eindeutig festgestellt werden.

Eine mögliche Erklärung wäre, dass der Agent beim Zurücksetzen der Episode direkt oder minimal mit einem Wall-Collider in Kontakt kommt. Auffällig ist allerdings, dass die Fälle nicht nur bei einem einzelnen Spawnpoint auftreten, sondern über verschiedene Runs und Spawnpoints verteilt sind. Daher wird hier nicht sicher behauptet, dass ausschließlich ein einzelner falsch platzierter Spawnpoint verantwortlich ist.

Die Werte werden nicht automatisch entfernt, da sie unter den tatsächlichen Testbedingungen entstanden sind und somit Teil der erhobenen Random Baseline sind. Zusätzlich wird geprüft, wie häufig diese Fälle auftreten und ob sich die zentralen Kennzahlen der Random Baseline verändern, wenn Episoden mit "steps = 0" ausgeschlossen werden. Dadurch kann eingeschätzt werden, ob diese Auffälligkeit die Interpretation der Baseline relevant beeinflusst.

Außerdem ist diese Erkenntnis auch für die spätere Analyse der trainierten Agenten relevant, da ähnliche Fälle auch dort auftreten könnten. Es wird daher wichtig sein, diese Fälle in den trainierten Agenten ebenfalls zu identifizieren und zu bewerten, um eine konsistente Analyse über alle Agenten hinweg zu gewährleisten.

**Anzahl der Fälle mit steps = 0:**

In [25]:
zero_step_episodes = df[df["steps"] == 0].copy()  # alle Episoden herausfiltern, die direkt bei Step 0 beendet wurden

zero_step_episodes.head()  # erste Beispiele anzeigen, um die Struktur der Fälle zu prüfen

,run_id,environment,algorithm,seed,episode,result,steps,total_reward,spawn_index,goal_index,max_steps,success,wall_hit,timeout,total_logged_steps
36,Random_Env1_Seed1,Env1_Simple,Random,1,37,Wall,0,-0.5,2,1,5000,0,1,0,18976
65,Random_Env1_Seed1,Env1_Simple,Random,1,66,Wall,0,-0.5,1,2,5000,0,1,0,34104
546,Random_Env1_Seed27,Env1_Simple,Random,27,147,Wall,0,-0.5,1,0,5000,0,1,0,81927
607,Random_Env1_Seed42,Env1_Simple,Random,42,8,Wall,0,-0.5,2,3,5000,0,1,0,4783
741,Random_Env1_Seed42,Env1_Simple,Random,42,142,Wall,0,-0.5,2,3,5000,0,1,0,76219


In [35]:
zero_step_count = len(zero_step_episodes)  # absolute Anzahl der Step-0-Episoden
total_episodes = len(df)  # Gesamtzahl aller Baseline-Episoden
zero_step_rate_total = zero_step_count / total_episodes  # Anteil der Step-0-Episoden an allen Episoden

print(f"Anzahl der Episoden mit steps = 0: {zero_step_count}" + f" von insgesamt {total_episodes} Episoden" + f" ({zero_step_rate_total:.2%} aller Episoden der Random Baseline)")

Anzahl der Episoden mit steps = 0: 32 von insgesamt 3000 Episoden (1.07% aller Episoden der Random Baseline)


In [36]:
zero_step_by_environment = (
    df.assign(is_zero_step=df["steps"] == 0)  # Hilfsspalte erstellen, ob eine Episode Step 0 hatte
      .groupby("environment")  # nach Environment gruppieren
      .agg(
          zero_step_count=("is_zero_step", "sum"),  # Anzahl der Step-0-Episoden
          episodes=("episode", "count"),  # Gesamtzahl der Episoden pro Environment
          zero_step_rate=("is_zero_step", "mean"),  # Anteil der Step-0-Episoden
      )
      .reset_index()  # wieder als normale Tabelle ausgeben
)

zero_step_by_environment  # Tabelle anzeigen

,environment,zero_step_count,episodes,zero_step_rate
0,Env1_Simple,8,1000,0.008
1,Env2_Complex,23,1000,0.023
2,Env3_Warehouse,1,1000,0.001


In [37]:
zero_step_by_spawn = (
    zero_step_episodes.groupby(["environment", "spawn_index"])  # nach Environment und Spawnpoint gruppieren
      .agg(
          zero_step_count=("episode", "count"),  # Anzahl der Step-0-Fälle zählen
      )
      .reset_index()  # wieder als normale Tabelle ausgeben
      .sort_values(["environment", "zero_step_count"], ascending=[True, False])  # auffällige Spawnpoints oben anzeigen
)

zero_step_by_spawn  # Ergebnis anzeigen

,environment,spawn_index,zero_step_count
1,Env1_Simple,2,4
0,Env1_Simple,1,2
2,Env1_Simple,3,2
3,Env2_Complex,0,6
4,Env2_Complex,1,4
5,Env2_Complex,2,4
6,Env2_Complex,3,3
9,Env2_Complex,6,2
10,Env2_Complex,7,2
7,Env2_Complex,4,1


In [38]:
df_without_zero_steps = df[df["steps"] > 0].copy()  # Datensatz ohne Step-0-Episoden erstellen

summary_with_zero_steps = (
    df.groupby("environment")  # Auswertung mit allen Episoden
      .agg(
          episodes=("episode", "count"),  # Anzahl Episoden
          success_rate=("success", "mean"),  # Erfolgsrate
          wall_rate=("wall_hit", "mean"),  # Wandkollisionsrate
          timeout_rate=("timeout", "mean"),  # Timeout-Rate
          mean_reward=("total_reward", "mean"),  # durchschnittlicher Reward
          mean_steps=("steps", "mean"),  # durchschnittliche Schrittzahl
      )
      .reset_index()
)

summary_without_zero_steps = (
    df_without_zero_steps.groupby("environment")  # Auswertung ohne Step-0-Episoden
      .agg(
          episodes=("episode", "count"),  # Anzahl Episoden nach Ausschluss
          success_rate=("success", "mean"),  # Erfolgsrate ohne Step-0-Fälle
          wall_rate=("wall_hit", "mean"),  # Wandkollisionsrate ohne Step-0-Fälle
          timeout_rate=("timeout", "mean"),  # Timeout-Rate ohne Step-0-Fälle
          mean_reward=("total_reward", "mean"),  # durchschnittlicher Reward ohne Step-0-Fälle
          mean_steps=("steps", "mean"),  # durchschnittliche Schrittzahl ohne Step-0-Fälle
      )
      .reset_index()
)

summary_zero_step_comparison = summary_with_zero_steps.merge(
    summary_without_zero_steps,
    on="environment",
    suffixes=("_all", "_without_zero_steps")
)  # beide Tabellen nebeneinander zusammenführen

summary_zero_step_comparison  # Vergleich anzeigen

,environment,episodes_all,success_rate_all,wall_rate_all,timeout_rate_all,mean_reward_all,mean_steps_all,episodes_without_zero_steps,success_rate_without_zero_steps,wall_rate_without_zero_steps,timeout_rate_without_zero_steps,mean_reward_without_zero_steps,mean_steps_without_zero_steps
0,Env1_Simple,1000,0.000,1.000,0.0,-0.997859,497.860,992,0.000000,1.000000,0.0,-1.001874,501.875000
1,Env2_Complex,1000,0.000,1.000,0.0,-0.612850,112.850,977,0.000000,1.000000,0.0,-0.615507,115.506653
2,Env3_Warehouse,1000,0.002,0.998,0.0,-0.619269,130.269,999,0.002002,0.997998,0.0,-0.619388,130.399399


---

## Zusammenfassung und Umgang mit den Random-Baseline-Ergebnissen

Die Random Baseline wurde für alle drei Environments mit fünf Seeds und jeweils 200 Episoden pro Run durchgeführt. Insgesamt wurden damit 3000 Episoden ausgewertet. Der Random Agent verwendete denselben diskreten Aktionsraum (Discrete(9)), dieselben Bewegungsparameter, dieselbe Spawn-/Goal-Randomisierung und dieselben Episode-Endbedingungen wie die später trainierten Agenten. Der Unterschied besteht darin, dass der Random Agent keine gelernte Policy nutzt, sondern seine Aktionen zufällig auswählt.

Die Ergebnisse zeigen deutlich, dass zufälliges Verhalten für die Navigationsaufgabe kaum geeignet ist. In Env1_Simple und Env2_Complex erreichte der Random Agent in keinem der getesteten Runs das Ziel. Alle Episoden endeten dort durch Wandkollisionen. In Env3_Warehouse wurden insgesamt nur zwei erfolgreiche Episoden beobachtet. Beide Erfolge traten bei derselben Spawn-Goal-Konstellation auf (spawn_index = 1, goal_index = 0). Beim Blick in die Videodokumentation zeigt sich, dass diese beiden Punkte räumlich relativ nah beieinander liegen. Dadurch ist es plausibel, dass der Agent in seltenen Fällen auch ohne gezielte Strategie zufällig das Ziel erreicht.

Diese Beobachtung ist für die Interpretation wichtig, führt aber nicht zu einer nachträglichen Änderung des Environment-Designs. Die Random Baseline und die später trainierten Agenten sollen unter denselben Bedingungen verglichen werden. Würde das Environment nach der Baseline angepasst, wäre die Vergleichbarkeit zwischen Random Agent und trainierten Agenten nicht mehr vollständig gegeben. Stattdessen wird die räumlich günstige Start-Ziel-Konstellation als methodischer Hinweis dokumentiert. Für zukünftige Experimente wäre es sinnvoll, Spawn- und Goalpunkte systematischer zu platzieren oder zusätzlich die Spawn-Goal-Distanzen zu erfassen.

Zusätzlich wurden Episoden mit steps = 0 untersucht. Diese Fälle bedeuten, dass eine Episode direkt beim Start beendet wurde, bevor der Agent eine reguläre Aktion ausführen konnte. Die genaue Ursache konnte im Rahmen der Prüfung nicht eindeutig festgestellt werden. Eine mögliche Erklärung wäre ein sehr enger oder kurzfristiger Collider-Kontakt beim Zurücksetzen des Agenten. Allerdings treten die Fälle über verschiedene Runs und unterschiedliche Spawnpoints verteilt auf. Einzelne visuell überprüfte Spawnpoints zeigten zudem ausreichend Abstand zu Wänden, weshalb die Ursache nicht eindeutig auf einen einzelnen falsch platzierten Spawnpoint zurückgeführt werden kann.

Die steps = 0-Episoden wurden daher nicht entfernt, sondern transparent dokumentiert. Zusätzlich wurde geprüft, ob ein Ausschluss dieser Episoden die zentralen Kennzahlen verändert. Der Vergleich zeigt, dass sich Success Rate, Wall Rate, Timeout Rate und Mean Reward nur minimal verändern. Auch die durchschnittliche Episodenlänge steigt nur leicht, wenn diese Episoden ausgeschlossen werden. Für die Hauptauswertung bleiben die Werte deshalb enthalten, da sie unter den tatsächlichen Testbedingungen entstanden sind und die Kernaussagen der Random Baseline nicht wesentlich beeinflussen.

Insgesamt bildet die Random Baseline damit eine sinnvolle untere Referenz für die spätere Bewertung der trainierten Algorithmen. Wenn PPO, A2C oder DQN deutlich bessere Erfolgsraten oder stabilere Reward-Werte erreichen, spricht das dafür, dass sie ein Navigationsverhalten gelernt haben, das über zufällige Aktionswahl hinausgeht. Gleichzeitig zeigt die Baseline, dass die Environments nicht trivial durch zufälliges Verhalten lösbar sind.